# Image Perceptual Losses Comparison Example

This notebook provides an example of using existing perceptual loss functions as objectives when optimizing lighting, as discussed in [section 3.1](https://scholarsarchive.byu.edu/cgi/viewcontent.cgi?article=12256&context=etd#page=19.1) of the thesis. A loop is configured to easily compare the effect of each of these criteria on the selected scene:
- SSIM (`SSIMLoss`)
- LPIPS (`LPIPSLoss`)
- VGG Style Transfer (`VGGStyleTransferLoss`)

In [ ]:
import os
import sys

if ".." not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

from examples.example_scenes import (
    BlenderManScene,
    CandleScene,
    CarScene,
    CarStudioScene,
    DinoScene,
    EinarScene,
    EinarSmallDomeScene,
    FlowerPotScene,
    HouseScene,
    RedCarScene,
    SciFiRobotScene,
    SpringPortraitScene,
    SpringPortraitSmallDomeScene,
    SpringScene,
)
from losses.image_image import (
    LPIPSLoss,
    SSIMLoss,
    VGGStyleTransferLoss,
)
from utils.color.linear_to_srgb_converters import LinearRec709ToAgXBase
from utils.color.tonemapping.agx_looks import AgXPunchyLook
from utils.model.model_utils import create_clip_model_and_tokenizer
from utils.optimize import optimize_with_criterion
from utils.record_keeping.experiment import FolderManager


In [ ]:
# Define scenes to benchmark
scenes_to_test = [
    SciFiRobotScene(device=device),
    CarScene(device=device),
]


In [ ]:
target_image_path = ""  # TODO: Update with reference image path

In [ ]:
# Hyperparameters
lr = 0.06
n_iter = 250
global_seed = 2
color_space_converter = LinearRec709ToAgXBase(AgXPunchyLook())

loss_classes_to_run = [
    SSIMLoss,
    LPIPSLoss,
    VGGStyleTransferLoss
]

use_fine_tuned_model = True

# Optimization Benchmark Loop
output_directory = "image_perceptual_example"

for scene in scenes_to_test:
    for loss_class in loss_classes_to_run:
        loss_name = loss_class.__name__
        print(f"--- Running {loss_name} on {scene.name} ---")
        criterion = loss_class(reference_image=target_image_path)
        model_name = loss_name

        optimize_with_criterion(
            scene,
            lr,
            n_iter,
            criterion,
            starting_multiplier_std=(0.1, 0.1, 0.1),
            output_subdirectory_name=output_directory,
            n_results=1,
            render_color_space_converter=color_space_converter,
            require_physically_plausible_multipliers=True,
            title_prefix=f"{loss_name} ({scene.name})",
            device=device,
            save_every=50,
            model_name=model_name,
            pretrained_source="",
            seed=global_seed,
        )